# Fraud Detection & AML Copilot

Banka işlemlerinde dolandırıcılığı (fraud) tespit eden ve şüpheli işlemler için otomatik Türkçe rapor taslağı üreten uçtan uca bir sistem.

**Akış:** Tespit (XGBoost) → Açıklama (SHAP) → Rapor (LLM)

- **Veri:** PaySim sentetik mobil ödeme veri seti.
- **Problem:** Fraud çok nadir (~%0.13) - dengesiz sınıflandırma. Amaç hem yakalamak hem de *neden* şüpheli olduğunu bir müfettişe açıklamak.
- **Yaklaşım:** Model şüpheliyi bulur, SHAP nedenini verir, LLM bunu resmi Türkçe rapora çevirir.

---
*An end-to-end system that detects fraudulent bank transactions and drafts a Turkish suspicious-transaction report for each. Flow: XGBoost (detect) → SHAP (explain) → LLM (report).*

## 0. Kurulum / Setup

Gerekli kütüphaneleri kuruyoruz. LLM adımı için Colab Secrets'a `GROQ_API_KEY` ekliyoruz.

In [7]:
!pip install -q xgboost shap groq

import pandas as pd
import numpy as np
import shap
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, average_precision_score, roc_auc_score

## 1. Veriyi tanıma / Exploring the data

Modele başlamadan önce veriyi anlıyoruz: boyut, sütunlar ve en önemlisi **fraud dengesi**. Fraud oranı çok düşükse bu bir *dengesiz sınıflandırma* problemidir — bu, hem model seçimini hem de değerlendirme metriğini belirler.

Ayrıca fraud'un hangi işlem tiplerinde olduğuna bakıyoruz. Bu veri setinde fraud **yalnızca TRANSFER ve CASH_OUT** işlemlerinde görülüyor; diğerlerinde (PAYMENT, CASH_IN, DEBIT) hiç yok. Mantıklı, çünkü dolandırıcı parayı hesaptan *çıkarır*.

*Before modeling, we check the shape, columns, and the fraud balance. Fraud appears only in TRANSFER and CASH_OUT — a fraudster moves money out — which lets us focus the model on those two types.*

In [6]:
df = pd.read_csv("/content/PS_20174392719_1491204439457_log.csv")  # dosya adını kendine göre değiştir

print("Boyut / Shape:", df.shape)
print("\nSütunlar / Columns:", df.columns.tolist())
print("\nFraud dengesi / Fraud balance:")
print(df["isFraud"].value_counts(normalize=True))
print("\nİşlem tipine göre fraud / Fraud by type:")
print(df.groupby("type")["isFraud"].sum())
df.head()

Boyut / Shape: (6362620, 11)

Sütunlar / Columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']

Fraud dengesi / Fraud balance:
isFraud
0    0.998709
1    0.001291
Name: proportion, dtype: float64

İşlem tipine göre fraud / Fraud by type:
type
CASH_IN        0
CASH_OUT    4116
DEBIT          0
PAYMENT        0
TRANSFER    4097
Name: isFraud, dtype: int64


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


## 2. Veri hazırlama ve özellik türetme / Data prep & feature engineering

**Neden sadece iki işlem tipi?** Fraud yalnızca TRANSFER ve CASH_OUT'ta olduğu için, diğer tipleri dahil etmek modeli fraud'un hiç olmadığı verilerle şişirir. Veriyi bu ikiye indiriyoruz.

**Neden yeni özellik türetiyoruz?** Normal bir işlemde "eski bakiye − yeni bakiye = işlem tutarı" olmalıdır. Tutmuyorsa bir tuhaflık vardır. Fraud işlemlerde hesap beklenmedik şekilde sıfırlanır; türettiğimiz `errorBalance` özellikleri tam olarak bunu yakalar.

**Neden ID sütunlarını atıyoruz?** Her işlemde farklı oldukları için genelleme sağlamaz, ezberlemeye (leakage) yol açabilir.

**Neden `type`'ı sayıya çeviriyoruz?** Modeller metinle değil sayılarla çalışır. İki tip kaldığı için basit ikili kodlama yeterli: TRANSFER = 1, CASH_OUT = 0.

*We keep only TRANSFER and CASH_OUT, derive balance-inconsistency features (fraud empties the account), drop ID columns (leakage risk), and binary-encode the type.*

In [8]:
# Sadece fraud'un olduğu iki tipe odaklan
df_model = df[df["type"].isin(["TRANSFER", "CASH_OUT"])].copy()

# Özellik türetme: bakiye tutarsızlıkları
df_model["errorBalanceOrig"] = df_model["oldbalanceOrg"] - df_model["newbalanceOrig"] - df_model["amount"]
df_model["errorBalanceDest"] = df_model["oldbalanceDest"] + df_model["amount"] - df_model["newbalanceDest"]

# type -> sayı (TRANSFER=1, CASH_OUT=0)
df_model["type_encoded"] = (df_model["type"] == "TRANSFER").astype(int)

features = ["amount", "oldbalanceOrg", "newbalanceOrig", "oldbalanceDest",
            "newbalanceDest", "errorBalanceOrig", "errorBalanceDest", "type_encoded"]
X = df_model[features]
y = df_model["isFraud"]

# Eksik etiketli satırları temizle
mask = y.notna()
X, y = X[mask], y[mask]

print("Boyut / Shape:", X.shape, "| Fraud:", int(y.sum()))
X.head()

Boyut / Shape: (2770409, 8) | Fraud: 8213


,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,errorBalanceOrig,errorBalanceDest,type_encoded
2,181.00,181.0,0.0,0.0,0.00,0.00,181.0,1
3,181.00,181.0,0.0,21182.0,0.00,0.00,21363.0,0
15,229133.94,15325.0,0.0,5083.0,51513.44,-213808.94,182703.5,0
19,215310.30,705.0,0.0,22425.0,0.00,-214605.30,237735.3,1
24,311685.89,10835.0,0.0,6267.0,2719172.89,-300850.89,-2401220.0,1


In [9]:
import pandas as pd

df = pd.read_csv("PS_20174392719_1491204439457_log.csv")

print("Boyut:", df.shape)
print("\nSütunlar:", df.columns.tolist())
print("\nFraud dengesi:")
print(df["isFraud"].value_counts())
print(df["isFraud"].value_counts(normalize=True))

print("\nİşlem tiplerine göre fraud sayısı:")
print(df.groupby("type")["isFraud"].sum())

print("\nİlk 5 satır:")
print(df.head())

Boyut: (6362620, 11)

Sütunlar: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']

Fraud dengesi:
isFraud
0    6354407
1       8213
Name: count, dtype: int64
isFraud
0    0.998709
1    0.001291
Name: proportion, dtype: float64

İşlem tiplerine göre fraud sayısı:
type
CASH_IN        0
CASH_OUT    4116
DEBIT          0
PAYMENT        0
TRANSFER    4097
Name: isFraud, dtype: int64

İlk 5 satır:
   step      type    amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0     1   PAYMENT   9839.64  C1231006815       170136.0       160296.36   
1     1   PAYMENT   1864.28  C1666544295        21249.0        19384.72   
2     1  TRANSFER    181.00  C1305486145          181.0            0.00   
3     1  CASH_OUT    181.00   C840083671          181.0            0.00   
4     1   PAYMENT  11668.14  C2048537720        41554.0        29885.86   

      nameDest  oldbalanceDest  newbalanceDest  isFra

## 3. Model: XGBoost

**Neden XGBoost?** Tablo (tabular) verisinde en güçlü modellerden biri. Karar ağaçlarını sırayla, her biri bir öncekinin hatasını düzelterek kurar; karmaşık, doğrusal olmayan ilişkileri yakalar. Bankacılık verisi tablo formatında olduğu için uygun.

**Neden `stratify`?** Fraud çok az olduğu için, rastgele bölersek fraud örnekleri bir tarafa toplanabilir. `stratify=y` fraud oranını hem eğitimde hem testte aynı tutar.

**Neden `scale_pos_weight`?** Dengesizliği modelin içinde çözer — fraud'u yanlış bilmenin cezasını normalden çok daha ağır yapar.

**Neden accuracy değil?** Dengesiz veride yanıltıcıdır ("hepsi normal" diyen model %99.9 alır ama tek fraud yakalamaz). Bunun yerine PR-AUC, recall (fraud'un ne kadarını yakaladık — kritik), precision (yanlış alarm).

*XGBoost for tabular data; stratify preserves the fraud ratio; scale_pos_weight handles imbalance inside the model; we evaluate with PR-AUC/recall/precision, not accuracy.*

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scale = (y_train == 0).sum() / (y_train == 1).sum()
print("scale_pos_weight:", round(scale, 1))

model = XGBClassifier(
    scale_pos_weight=scale, max_depth=5, n_estimators=200,
    learning_rate=0.1, random_state=42, eval_metric="aucpr")
model.fit(X_train, y_train)

y_proba = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

print("PR-AUC :", round(average_precision_score(y_test, y_proba), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_proba), 4))
print(classification_report(y_test, y_pred, digits=4))

scale_pos_weight: 336.3
PR-AUC : 0.9974
ROC-AUC: 0.9983
              precision    recall  f1-score   support

           0     1.0000    0.9999    1.0000    552439
           1     0.9808    0.9970    0.9888      1643

    accuracy                         0.9999    554082
   macro avg     0.9904    0.9984    0.9944    554082
weighted avg     0.9999    0.9999    0.9999    554082



## 4. Açıklanabilirlik: SHAP / Explainability

XGBoost güçlü ama opaktır bir işlemin *neden* şüpheli işaretlendiğini tek başına söylemez. Bir müfettişin ve regülasyonun bu açıklamaya ihtiyacı vardır.

**SHAP** her tahmini, her özelliğin katkısına ayırır (pozitif = riski artırır). İki amaçla kullanıyoruz: (1) modelin tek bir özelliğe mi yoksa birden çok sinyale mi dayandığını *doğrulamak* (leakage kontrolü), (2) tek bir işlem için "neden şüpheli" faktörlerini çıkarıp LLM'e vermek.

*SHAP splits each prediction into per-feature contributions — used both to validate the model isn't leaning on one leaky feature and to extract the "why" for a single transaction.*

In [11]:
explainer = shap.TreeExplainer(model)
X_sample = X_test.iloc[:2000]
shap_values = explainer.shap_values(X_sample)

importance = np.abs(shap_values).mean(axis=0)
print("Genel özellik önemi / Overall feature importance:")
for feat, imp in sorted(zip(X.columns, importance), key=lambda x: -x[1]):
    print(f"  {feat:20s} {imp:.4f}")

Genel özellik önemi / Overall feature importance:
  errorBalanceOrig     7.5376
  oldbalanceDest       1.6565
  oldbalanceOrg        1.3738
  newbalanceOrig       1.0835
  amount               0.7749
  newbalanceDest       0.6904
  errorBalanceDest     0.2945
  type_encoded         0.2393


## 5. Bir şüpheli işlemi inceleme / Inspecting one suspicious transaction

En yüksek fraud olasılıklı işlemi seçip SHAP faktörlerini çıkarıyoruz. Bu, bir müfettişin görmek isteyeceği bilgidir: işlemin detayları + onu şüpheli yapan faktörler. Bu çıktı, bir sonraki adımda LLM'e vereceğimiz girdidir.

*We pick the highest-risk transaction and extract its SHAP factors — the input for the LLM report.*

In [12]:
y_proba_all = model.predict_proba(X_test)[:, 1]
idx = y_proba_all.argmax()

islem = X_test.iloc[idx]
olasilik = y_proba_all[idx]

print(f"Fraud olasılığı / probability: {olasilik:.4f}")
print(f"Gerçekte fraud mu / actually fraud: {int(y_test.iloc[idx])}")
print("\nİşlem / Transaction:")
print(islem)

shap_tek = explainer.shap_values(X_test.iloc[[idx]])[0]
print("\nSHAP katkıları (pozitif = risk artırır):")
for feat, val in sorted(zip(X.columns, shap_tek), key=lambda x: -abs(x[1])):
    print(f"  {feat:20s} {val:+.3f}")

Fraud olasılığı / probability: 1.0000
Gerçekte fraud mu / actually fraud: 1

İşlem / Transaction:
amount              7703574.71
oldbalanceOrg       7703574.71
newbalanceOrig            0.00
oldbalanceDest            0.00
newbalanceDest            0.00
errorBalanceOrig          0.00
errorBalanceDest    7703574.71
type_encoded              1.00
Name: 5188008, dtype: float64

SHAP katkıları (pozitif = risk artırır):
  errorBalanceOrig     +9.820
  amount               +2.319
  newbalanceDest       +1.191
  oldbalanceDest       +0.479
  newbalanceOrig       +0.474
  oldbalanceOrg        +0.324
  errorBalanceDest     +0.213
  type_encoded         -0.155


## 6. LLM ile Türkçe Şüpheli İşlem Raporu / Turkish report via LLM

Projenin operasyonel değeri burada. Klasik bir model sadece "bu işlem şüpheli" der. Biz bir adım öteye gidip SHAP faktörlerini, bir müfettişin inceleyip onaylayacağı **resmi Türkçe rapor taslağına** çeviriyoruz.

İşlem detayları + risk faktörleri bir LLM'e (Groq üzerinden Llama-3.3) veriliyor; modelin halüsinasyon görmesini engellemek ve deterministik, kurumsal bir çıktı almak için temperature=0.1 kullanılıyor ve modele katı bir Markdown şablonu zorunlu tutuluyor. Böylece her çalıştırmada teknik terimlerden arındırılmış, standartlara uygun ve anında aksiyon alınabilecek netlikte bir rapor üretiliyor. "Tamamen Türkçe yaz" talimatı, modelin başka dile kaymasını engeller.

*The true operational value lies here. Instead of merely flagging a transaction as "suspicious," we pass the SHAP risk factors to an LLM (Llama-3.3 via Groq) constrained by a strict Markdown template. Using temperature=0.1 ensures a deterministic, highly structured, and jargon-free Turkish Suspicious Activity Report (SAR) draft — directly reducing the investigator's manual workload and enabling immediate action.*

In [14]:
from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get("GROQ_API_KEY"))

islem_bilgisi = f"""
İşlem Tipi: {'TRANSFER' if islem['type_encoded']==1 else 'CASH_OUT'}
İşlem Tutarı: {islem['amount']:,.2f} TL
Gönderen - İşlem öncesi bakiye: {islem['oldbalanceOrg']:,.2f} TL
Gönderen - İşlem sonrası bakiye: {islem['newbalanceOrig']:,.2f} TL
Alıcı - İşlem öncesi bakiye: {islem['oldbalanceDest']:,.2f} TL
Alıcı - İşlem sonrası bakiye: {islem['newbalanceDest']:,.2f} TL
Fraud olasılığı: {olasilik:.1%}

En etkili risk faktörleri:
- Gönderen bakiye tutarsızlığı: en güçlü sinyal
- Yüksek işlem tutarı
- Gönderen hesabının sıfırlanması
"""

prompt = f"""Sen bir bankanın kara para aklama (AML) müfettişine yardımcı olan kıdemli bir asistansın.
Aşağıdaki işlem bilgilerini inceleyip, müfettişin paneline düşecek resmi bir Şüpheli İşlem Raporu (ŞİB) üret.
Aşağıdaki MARKDOWN ŞABLONUNA BİREBİR uyarak yanıt ver. Şablonun dışına çıkma, kendi kendine giriş veya kapanış cümlesi ekleme.

İŞLEM BİLGİLERİ:
{islem_bilgisi}

MARKDOWN ŞABLONU:
🚨 ŞÜPHELİ İŞLEM BİLDİRİMİ (ŞİB) - RİSK SEVİYESİ: [Kritik/Yüksek]

 📌 İşlem Özeti
İşlem Tipi: [İşlem Tipi]
İşlem Tutarı: [Tutar]
AI Fraud Skoru: [Olasılık Yüzdesi]

 💳 Hesap Hareketleri
Gönderen Hesap: İşlem öncesi: [Tutar] ➔ İşlem sonrası: [Tutar]
Alıcı Hesap: İşlem öncesi: [Tutar] ➔ İşlem sonrası: [Tutar]

 ⚠️ Tespit Edilen Risk Faktörleri
 [Risk 1 - Açıklamasıyla birlikte]
 [Risk 2 - Açıklamasıyla birlikte]
 [Risk 3 - Açıklamasıyla birlikte]

> 📝 AI Aksiyon Önerisi:
> [Risk faktörlerine dayanarak müfettişe sunulan, 'İşlem derhal bloke edilmeli', 'Detaylı incelemeye alınmalı' gibi tek cümlelik net aksiyon önerisi]
"""

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt}],
    temperature=0.1,
)
print(response.choices[0].message.content)

🚨 ŞÜPHELİ İŞLEM BİLDİRİMİ (ŞİB) - RİSK SEVİYESİ: Kritik

 📌 İşlem Özeti
İşlem Tipi: TRANSFER
İşlem Tutarı: 7,703,574.71 TL
AI Fraud Skoru: 100.0%

 💳 Hesap Hareketleri
Gönderen Hesap: İşlem öncesi: 7,703,574.71 TL ➔ İşlem sonrası: 0.00 TL
Alıcı Hesap: İşlem öncesi: 0.00 TL ➔ İşlem sonrası: 0.00 TL

 ⚠️ Tespit Edilen Risk Faktörleri
- Gönderen bakiye tutarsızlığı: Gönderen hesabının işlem öncesi ve sonrası bakiyeleri arasında tutarsızlık mevcut, bu durum yüksek risk oluşturuyor.
- Yüksek işlem tutarı: İşlem tutarı çok yüksek, bu da potansiyel bir kara para aklama faaliyetine işaret ediyor.
- Gönderen hesabının sıfırlanması: Gönderen hesabının işlem sonrası bakiyesinin sıfırlanması, hesabın possibly kötü niyetli amaçlar için kullanıldığını gösteriyor.

> 📝 AI Aksiyon Önerisi:
> İşlem derhal bloke edilmeli ve detaylı bir şekilde incelenmelidir.




## 7. LangGraph Agent Workflow / Ajan İş Akışı

Şu ana kadar adımlar tek tek çalıştı: skorla → SHAP → rapor. Bu bölümde onları bir **LangGraph state machine**'e bağlıyoruz — adımlar birbirine akıyor ve ajan **risk-bazlı koşullu karar** veriyor.

Akış: **fetch (işlemi al) → analyze (model + SHAP) → koşullu dal:**
- Fraud olasılığı yüksekse (> 0.80) → rapor üret → müfettişe işaretle
- Düşükse → otomatik onayla (rapor üretme)

Bu, gerçek bir AML sürecinin mantığıdır: her işleme rapor yazılmaz, sadece riskli olanlar müfettişe (human-in-the-loop) gider. Ajan bu kararı kendisi verir.

*The detection, explanation, and reporting steps are wired into a LangGraph state machine with risk-based routing: high-risk transactions get a report and are flagged for a human investigator; low-risk ones are auto-approved.*

In [15]:
from langgraph.graph import StateGraph, END
from typing import TypedDict

# --- Ajanın durumu (state): düğümler arası taşınan bilgi ---
class FraudState(TypedDict):
    idx: int              # incelenecek işlemin indeksi
    probability: float    # model fraud olasılığı
    shap_factors: str     # SHAP'tan gelen risk faktörleri
    decision: str         # "REPORTED" veya "AUTO_APPROVED"
    report: str           # LLM rapor taslağı (varsa)

# --- Düğüm 1: fetch — işlemi al ---
def fetch(state: FraudState) -> FraudState:
    i = state["idx"]
    state["probability"] = float(model.predict_proba(X_test.iloc[[i]])[:, 1][0])
    return state

# --- Düğüm 2: analyze — SHAP faktörlerini çıkar ---
def analyze(state: FraudState) -> FraudState:
    i = state["idx"]
    shap_vals = explainer.shap_values(X_test.iloc[[i]])[0]
    top = sorted(zip(X.columns, shap_vals), key=lambda x: -abs(x[1]))[:3]
    state["shap_factors"] = ", ".join(f"{f} ({v:+.2f})" for f, v in top)
    return state

# --- Koşullu yönlendirme: risk yüksek mi? ---
def route(state: FraudState) -> str:
    return "high_risk" if state["probability"] > 0.80 else "low_risk"

# --- Düğüm 3a: yüksek risk → LLM rapor üret ---
def generate_report(state: FraudState) -> FraudState:
    islem = X_test.iloc[state["idx"]]
    bilgi = f"""İşlem Tipi: {'TRANSFER' if islem['type_encoded']==1 else 'CASH_OUT'}
İşlem Tutarı: {islem['amount']:,.2f}
Gönderen bakiye: {islem['oldbalanceOrg']:,.2f} -> {islem['newbalanceOrig']:,.2f}
Fraud olasılığı: {state['probability']:.1%}
Risk faktörleri: {state['shap_factors']}"""

    prompt = f"""Sen bir bankanın AML müfettişine yardımcı olan bir asistansın.
Aşağıdaki şüpheli işlem için resmi, kısa bir Şüpheli İşlem Raporu taslağı yaz.
Teknik terim kullanma. Raporu tamamen Türkçe yaz, başka dilde karakter kullanma.

{bilgi}

Rapor taslağı:"""

    resp = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
    )
    state["report"] = resp.choices[0].message.content
    state["decision"] = "REPORTED"
    return state

# --- Düğüm 3b: düşük risk → otomatik onay ---
def auto_approve(state: FraudState) -> FraudState:
    state["decision"] = "AUTO_APPROVED"
    state["report"] = "Düşük risk — otomatik onaylandı, rapor gerektirmez."
    return state

# --- Grafiği kur ---
graph = StateGraph(FraudState)
graph.add_node("fetch", fetch)
graph.add_node("analyze", analyze)
graph.add_node("generate_report", generate_report)
graph.add_node("auto_approve", auto_approve)

graph.set_entry_point("fetch")
graph.add_edge("fetch", "analyze")
graph.add_conditional_edges("analyze", route,
    {"high_risk": "generate_report", "low_risk": "auto_approve"})
graph.add_edge("generate_report", END)
graph.add_edge("auto_approve", END)

agent = graph.compile()
print("Ajan kuruldu / Agent compiled.")

Ajan kuruldu / Agent compiled.


In [16]:
# En yüksek riskli işlemi bul (yüksek risk dalını test eder)
y_proba_all = model.predict_proba(X_test)[:, 1]
yuksek_idx = int(y_proba_all.argmax())

# Düşük riskli bir işlem bul (düşük risk dalını test eder)
dusuk_idx = int(y_proba_all.argmin())

for isim, idx in [("YÜKSEK RİSK", yuksek_idx), ("DÜŞÜK RİSK", dusuk_idx)]:
    print("="*60)
    print(f"--- {isim} İŞLEM (idx={idx}) ---")
    sonuc = agent.invoke({"idx": idx, "probability": 0, "shap_factors": "",
                          "decision": "", "report": ""})
    print(f"Fraud olasılığı: {sonuc['probability']:.1%}")
    print(f"Karar: {sonuc['decision']}")
    print(f"Rapor:\n{sonuc['report']}")
    print()

--- YÜKSEK RİSK İŞLEM (idx=3202) ---
Fraud olasılığı: 100.0%
Karar: REPORTED
Rapor:
Şüpheli İşlem Raporu Taslağı

Rapor Tarihi: [Gün/Ay/Yıl]
Rapor No: [Rapor Numarası]

Şüpheli İşlem Bilgileri:
- İşlem Tipi: Transfer
- İşlem Tutarı: 7.703.574,71
- Gönderen Bakiye Öncesi: 7.703.574,71
- Gönderen Bakiye Sonrası: 0,00

Şüpheli İşlem Nedenleri:
- Fraud olasılığı yüksek olarak belirlenmiştir (%100).
- Risk faktörleri:
  - errorBalanceOrig (+9,82)
  - amount (+2,32)
  - newbalanceDest (+1,19)

Bu işlem, yüksek bir fraud olasılığına sahip olup, risk faktörleri de bu şüpheli durumu desteklemektedir. İşlem tutarı ve gönderen bakiyenin aniden sıfıra düşmesi, ayrıca belirtilen risk faktörleri, bu işlemin daha detaylı bir şekilde incelenmesini gerektirmektedir.

İnceleme ve gerektiği durumlarda ilgili mercilere bildirilmesi önerilir.

--- DÜŞÜK RİSK İŞLEM (idx=44719) ---
Fraud olasılığı: 0.0%
Karar: AUTO_APPROVED
Rapor:
Düşük risk — otomatik onaylandı, rapor gerektirmez.



## Bilinen sınırlar / Known limitations

- **Sentetik veri / Synthetic data.** PaySim gerçek değil; gerçek fraud desenleri daha karmaşıktır. Sonuçlar bu veri setine özgüdür.
- **Türetilen bakiye özellikleri güçlü sinyal veriyor** — yüksek skoru kısmen açıklar; gerçek dünyada tek bir özellik bu kadar belirleyici olmayabilir.
- **LLM raporu taslaktır, resmi belge değildir** / a draft, not an official document — müfettiş incelemesi gerektirir.
- **Tek train/test split**, hiperparametre optimizasyonu yok — çalışan prototip, tune edilmiş production modeli değil.